In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pywt

In [ ]:
try:
    noisy_signal = pd.read_csv('noisy_signal_1.csv', header=None).values.flatten()
    transient = pd.read_csv('transient1.csv', header=None).values.flatten()
except FileNotFoundError:
    pass

In [ ]:
def denoise_signal(data, wavelet='db4', level=1):
    coeffs = pywt.wavedec(data, wavelet, mode='per')
    sigma = (1/0.6745) * np.median(np.abs(coeffs[-1] - np.median(coeffs[-1])))
    uthresh = sigma * np.sqrt(2 * np.log(len(data)))
    coeffs_thresh = [coeffs[0]] + [pywt.threshold(c, value=uthresh, mode='soft') for c in coeffs[1:]]
    return pywt.waverec(coeffs_thresh, wavelet, mode='per')

denoised_signal = denoise_signal(noisy_signal, wavelet='sym4', level=2)

plt.figure(figsize=(12, 5))
plt.plot(noisy_signal, label='Original Noisy Signal', color='lightgray', alpha=0.8)
plt.plot(denoised_signal, label='Denoised Signal (Wavelet)', color='blue', linewidth=1.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
wavelet_type = 'cmor1.5-1.0'
scales = np.arange(1, 128)

coefficients, frequencies = pywt.cwt(transient, scales, wavelet_type)
power = (np.abs(coefficients)) ** 2

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2]})

ax1.plot(transient, color='red')
ax1.set_xlim([0, len(transient)-1])

im = ax2.imshow(power, extent=[0, len(transient)-1, scales[-1], scales[0]], 
                aspect='auto', cmap='jet', interpolation='bilinear')
fig.colorbar(im, ax=ax2)

plt.tight_layout()
plt.show()